# Crosstabs Algorithm

In [ ]:
# The crosstabs algorithm computes the contingency table of two or more categorical
# variables. I suggest to have a brief look at the swimlane diagram:
# https://algorithms.vantage6.ai/en/latest/v6-crosstab-py/docs/v6-crosstab-py/implementation.html#overview
# to have a good overview of the different steps in the algorithm. From the diagram
# you can see that this is a single (federated-)step algorithm:
#
# 1. Call `partial_crosstab`
#
# And then there is the central part responsible for the aggregation of the results. The
# central part of the algorithm (the main call) will return the contingency table for
# the entire federated dataset. 
#
# 1. Create a new vantage6 task to execute the *crosstabs* method (central part). This
#    central part will start the tasks `partial_crosstab` (as you can see in the 
#    swimlane diagram).
# 2. Poll until the task is finished
# 3. Retrieve the *global* contingency table from the central part (the main call)
# 4. Retrieve the *local* contingency table from the data stations (the
#    `partial_crosstab` call that was made by the central part)
#

In [ ]:
import base64
import json
import requests

In [ ]:
with open("token.txt", "r") as f:
    token = f.read().strip()
headers = {
    "Authorization": token
}

In [ ]:
# Set the collaboration ID:
#
#   - 2: Test collaboration (with IKNL and UPM)
#   - 3: IDEA4RC collaboration
#
COLLABORATION_ID = 3

In [ ]:
# Select the organization IDs that can be selected by the user. These should be the IDs 
# of the vantage6 organizations:
#
#   1	- root
#   2	- ENG
#   3	- UPM
#   4	- INT
#   5	- UKE
#   6	- CLB
#   7	- FPNS
#
ORGANIZATION_IDS = [1, 4]

In [ ]:
# Set the study ID. This is the `study` id that belongs to the RAVEN workspace. See the
# `0-new-workspace.ipynb` notebook for more information.
STUDY_ID = 16
# Set the session ID. This is the `v6_session` id that belongs to the RAVEN analysis.
# See the `1-new-analysis.ipynb` notebook for more information.
SESSION_ID = 7

In [ ]:
# The image to use is the latest version of the analytics algorithm
IMAGE = "harbor2.vantage6.ai/idea4rc/analytics:latest"
#
# The method (that is within this IMAGE) to execute is the summary algorithm. For
# data exploration we use the `summary` method.
METHOD = "crosstab"

In [ ]:
# The user is able to select multiple cohorts in RAVEN. These cohorts correspond to
# different dataframes, see the `2-new-cohort.ipynb` how these are created and check
# the `v6_dataframe` column in the RAVEN database for more information.
#
# In this example I've just used one cohort. Simply add more ids to the list to use
# more cohorts.
DATAFRAME_IDS = [132]

In [ ]:
# Before we can start analysis the cohorts (dataframes) we need to check the variables
# that are available in the dataframes. You can use this endpoint whenever you want user
# to allow you to select variables.
# TODO the dtpyes might change in the future, so do not rely on them to heavily now. In
# the next version of the data extraction job we will likely provide you with either the
# `category` or `numeric` colum type (so that you can use them to select varables)
response = requests.get(
    f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/session/dataframe/{DATAFRAME_IDS[0]}",
    headers=headers
)
VARIABLES = response.json()["columns"]
VARIABLES

In [ ]:
# The user is able to select two variables to compute the contingency table of.
RESULTS_COL = "sex"
# NOTE: This is where you add the group columns which can be one or more columns
GROUP_COLS = ["clinical_stage", "pathological_stage"] 

In [ ]:
org_input = [
    {
        "id": ORGANIZATION_IDS[0], # Central task
        "arguments": base64.b64encode(
            json.dumps(
                {
                    "results_col": RESULTS_COL,
                    "group_cols": GROUP_COLS,
                    # user selected participants
                    "organizations_to_include": ORGANIZATION_IDS 
                }
            ).encode("UTF-8")
        ).decode("UTF-8")
    }
]

payload = {
    "name": "Human-readable name of the task",
    "image": IMAGE,
    "description": "Description of the task",
    "action": "central_compute",
    "method": METHOD,
    "organizations": org_input,
    "databases": [
        [
            {
                "type": "dataframe",
                "dataframe_id": df_id
            } for df_id in DATAFRAME_IDS
        ]
    ],
    "session_id": SESSION_ID,
    "study_id": STUDY_ID
}
payload

In [ ]:
# Then using the authorization header and the payload we can create a vantage6 task
# using the vantage6 server API.
response = requests.post(
    "https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/task",
    headers=headers,
    json=payload
)
# In the response we need to extract the task ID and the job ID so we can poll werther
# the (central) task is finished. Later on we can also use these IDs to retrieve the
# results.
TASK_ID = response.json()["id"]
JOB_ID = response.json()["job_id"]
response.json()

In [ ]:
# Poll until the (central) task is finished. We do not concern about the subtasks in
# this instance. We could consider including them in the future as well?
response = requests.get(
    f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/run?task_id={TASK_ID}",
    headers=headers,
)
# Since it is a central task we can obtain the [0]th element of the data list as it
# always should be a single element in a list. Wait until the status returns
# "completed".
response.json()["data"][0]["status"]

In [ ]:
# Get the results of the (central) task, thus the *global* summary statistics.
response_global = requests.get(
    f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/result?task_id={TASK_ID}",
    headers=headers,
)
# Again, since it is a central task we can obtain the [0]th element of the data list
result_global = [json.loads(base64.b64decode(result["result"]).decode("UTF-8")) for result in response_global.json()["data"]]
result_global

# Visualization of results

The function <code>visualize_contingency()</code> is responsible for presenting the outcomes of the crosstabs algorithm in an intuitive way. It performs two key tasks:

1. <b>Displays Statistical Results</b><br>
The function prints the chi-squared statistic and the corresponding p-value, giving a quick indication of whether there is a significant association between the variables.


2. <b>Generates Cohort-Specific Tables</b><br>
For each cohort, a contingency table is created and visualized, making it easy to compare distributions across groups.


An additional argument, <code>percentage_mode</code>, allows the user to control how percentages are displayed in the tables. The available options are:

* <b>row</b> – Percentages are calculated within each row.
* <b>column</b> – Percentages are calculated within each column.
* <b>total</b> – Percentages are based on the overall total.

This flexibility ensures that the visualization can be tailored to the analytical perspective most relevant to the user.

## Function

In [ ]:
import pandas as pd
import numpy as np

def visualize_contingency(result, row_var, col_var, percentage_mode=None):
    """
    Visualize contingency tables from the given result structure with counts + percentages.

    Parameters
    ----------
    result : list or dict
        JSON-like structure containing cohort data.
    row_var : str
        Variable to use for rows (e.g., "FNCLCC_GRADE", "Sex").
    col_var : str
        Variable to use for columns (e.g., "Histology", "Tumor Rupture").
    percentage_mode: str, optional
        Mode for displaying percentages ("row", "column", "total").
    """

    # Handle list-of-dict input
    if isinstance(result, list) and isinstance(result[0], dict):
        result = result[0]

    if not isinstance(result, dict):
        raise ValueError("Expected 'result' to be a dict or a list containing one dict")

    for cohort_name, cohort_data in result.items():
        contingency = cohort_data.get("contingency_table", [])
        if not contingency:
            print(f"⚠️ No contingency table found for {cohort_name}")
            continue

        df = pd.DataFrame(contingency)

        # Identify row column
        row_col_name = None
        for c in df.columns:
            if row_var.lower() in c.lower():
                row_col_name = c
                break
        if row_col_name is None:
            print(f"⚠️ Could not find column for '{row_var}' in {cohort_name}")
            continue

        df.rename(columns={row_col_name: row_var.replace("_", " ").title()}, inplace=True)
        row_col_name = row_var.replace("_", " ").title()

        # Convert numeric columns safely
        for c in df.columns:
            if c != row_col_name:
                df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

        # Sort columns (alphabetical except 'N/A' and 'Total' at end)
        cols = sorted([c for c in df.columns if c not in [row_col_name, 'N/A', 'Total']])
        for special in ['N/A', 'Total']:
            if special in df.columns:
                cols.append(special)
        df = df[[row_col_name] + cols]

        # --- Percentage computation ---
        if percentage_mode is not None:
            numeric_cols = [c for c in df.columns if c != row_col_name]
            df_pct = pd.DataFrame(index=df.index, columns=numeric_cols, dtype=float)

            total_row_mask = df[row_col_name].astype(str).str.lower() == "total"
            data_rows_mask = ~total_row_mask

            for col in numeric_cols:
                if percentage_mode == "row":
                    for idx in df.index:
                        row_total = df.loc[idx, "Total"] if "Total" in df.columns else np.nan
                        if col == "Total":
                            df_pct.loc[idx, col] = 100
                        else:
                            df_pct.loc[idx, col] = df.loc[idx, col] / row_total * 100 if pd.notna(row_total) and row_total > 0 else 0

                elif percentage_mode == "column":
                    # Column total excluding Total row
                    col_total = df.loc[data_rows_mask, col].sum()
                    for idx in df.index:
                        if total_row_mask.loc[idx]:
                            df_pct.loc[idx, col] = np.nan
                        else:
                            df_pct.loc[idx, col] = df.loc[idx, col] / col_total * 100 if col_total > 0 else 0
                    # Total row percentage = 100%
                    if total_row_mask.any():
                        df_pct.loc[total_row_mask, col] = 100.0

                elif percentage_mode == "total":
                    # Grand total excluding Total row/col
                    grand_total = df.loc[data_rows_mask, [c for c in df.columns if c != row_col_name and c != "Total"]].to_numpy().sum()
                    for idx in df.index:
                        df_pct.loc[idx, col] = df.loc[idx, col] / grand_total * 100 if grand_total > 0 else 0

                else:
                    raise ValueError("percentage_mode must be 'row', 'column', or 'total'")

            # --- Combine counts + percentages ---
            df_pct = df_pct.fillna(0)
            for col in numeric_cols:
                df[col] = df[col].astype(int).astype(str) + " (" + df_pct[col].round(1).astype(str) + "%)"

        # --- MultiIndex for display ---
        multi_cols = [(col_var.replace("_", " ").title(), c) if c not in [row_col_name, "Total"]
                      else ("", c) for c in df.columns]
        df.columns = pd.MultiIndex.from_tuples(multi_cols)

        # --- Styling ---
        is_sub = df[("", row_col_name)].astype(str).str.lower().str.startswith("grade")

        def style_rows(row):
            if not is_sub.loc[row.name]:
                return ["font-weight: normal;"] * len(row)
            else:
                return ["padding-left: 20px"] + [""] * (len(row) - 1)

        styled_df = (
            df.style
            .apply(style_rows, axis=1)
            .hide(axis="index")
            .set_properties(subset=[("", row_col_name)], **{"text-align": "left"})
            .set_table_styles([
                {"selector": "th", "props": [
                    ("font-weight", "bold"),
                    ("text-align", "center")
                ]}
            ])
        )

        # --- Chi-square stats ---
        chi2 = cohort_data.get("chi2", {}).get("chi2")
        pval = cohort_data.get("chi2", {}).get("P-value")

        print(f"\n--- {cohort_name.replace('_', ' ').title()} ---")
        if chi2 and pval:
            print(f"Chi² = {float(chi2):.3f},  p = {float(pval):.3f}")
        if percentage_mode:
            print(f"Percentage mode: {percentage_mode.title()}")

        display(styled_df)


## Output

In [ ]:
visualize_contingency(result_global[0], row_var=GROUP_COLS[0], col_var=RESULTS_COL, percentage_mode="row")

In [ ]:
visualize_contingency(result_global[0], row_var=GROUP_COLS[0], col_var=RESULTS_COL, percentage_mode="column")

In [ ]:
visualize_contingency(result_global[0], row_var=GROUP_COLS[0], col_var=RESULTS_COL, percentage_mode="total")